# Robustness Checks

## Setup

In [1]:
import os
import pickle
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from config import BATCH_SIZE, SEED
from src.dataloader import get_dataloaders
from src.model import DissagreementPredictor
from src.utils import set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FIGURES_DIR = PROJECT_ROOT / "figures"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
DEVICE

device(type='cpu')

In [2]:
def load_cifar_batch(batch_path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with open(batch_path, "rb") as f:
            return pickle.load(f, encoding="bytes")


batch = load_cifar_batch("data/raw/cifar-10-batches-py/test_batch")

images = batch[b"data"]
hard_labels_all = np.array(batch[b"labels"])

probs = np.load("data/raw/cifar10h-probs.npy")
test_idx = np.load("data/processed/test_idx.npy")

_, _, test_loader = get_dataloaders(images, probs, batch_size=BATCH_SIZE)

hard_labels = hard_labels_all[test_idx]

## Check A — Annotator Subsampling

In [3]:
model = DissagreementPredictor().to(DEVICE)
model.load_state_dict(torch.load("checkpoints/best_model_custom.pt", map_location=DEVICE))
model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for imgs, targets in test_loader:
        preds = model(imgs.to(DEVICE))
        all_preds.append(preds.cpu().numpy())
        all_true.append(targets.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

all_preds.shape, all_true.shape

((2000, 10), (2000, 10))

## Check B — OOD Corruption

In [4]:
def mean_entropy(p):
    p = np.clip(p, 1e-8, 1)
    return np.mean(-np.sum(p * np.log(p), axis=1))


def gaussian_noise(x, severity):
    std = [0.04,0.08,0.12,0.18,0.26][severity-1]
    return torch.clamp(x + torch.randn_like(x)*std, 0, 1)


def contrast(x, severity):
    factor = [0.8,0.6,0.4,0.2,0.1][severity-1]
    mean = x.mean(dim=(2,3), keepdim=True)
    return torch.clamp((x-mean)*factor + mean, 0, 1)


corruptions = {
    "noise": gaussian_noise,
    "contrast": contrast
}

In [ ]:
subset = []
for imgs, _ in test_loader:
    subset.append(imgs)
    if len(subset)*imgs.size(0) > 1000:
        break

subset = torch.cat(subset)[:1000].to(DEVICE)

results = {}

for name, fn in corruptions.items():
    values = []
    for s in range(1,6):
        with torch.no_grad():
            out = model(fn(subset, s))
            values.append(mean_entropy(out.cpu().numpy()))
    results[name] = values
    

In [ ]:
plt.figure()

for k,v in results.items():
    plt.plot(range(1,6), v, marker="o", label=k)

plt.legend()
plt.title("Entropy vs Corruption")
plt.xlabel("Severity")
plt.ylabel("Entropy")

plt.savefig("figures/robustness_entropy.png")
plt.show()

## Check C — Class-conditional Performance

In [ ]:
def compute_jsd(p,q):
    p = np.clip(p,1e-8,1)
    q = np.clip(q,1e-8,1)
    m = 0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m),1) + 0.5*np.sum(q*np.log(q/m),1)


CLASS_NAMES = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

jsd_scores = []

for i in range(10):
    mask = hard_labels == i
    jsd = compute_jsd(all_preds[mask], all_true[mask]).mean()
    jsd_scores.append(jsd)
    print(CLASS_NAMES[i], jsd)

In [ ]:
plt.figure()

plt.barh(CLASS_NAMES, jsd_scores)

plt.title("Class-wise JSD")

plt.savefig("figures/classwise_jsd.png")
plt.show()